# Phase 3: Model Explainability with SHAP

While gradient boosting models like LightGBM are highly accurate, they are often considered 'black boxes'. In the insurance industry, interpretability is crucial for regulatory compliance and business trust.

In this notebook, we use **SHAP (SHapley Additive exPlanations)** to unpack our LightGBM Claim Frequency model and understand exactly which features drive risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')

## 1. Load Model and Data
We load the best frequency model saved from our pipeline and select a sample of test data to explain.

In [ ]:
# Load model pipeline
with open('../reports/lgbm_frequency_model.pkl', "rb") as f:
    pipeline = pickle.load(f)
    
preprocessor = pipeline.named_steps["preprocessor"]
lgb_model = pipeline.named_steps["model"]

# Load data and sample 5000 rows for SHAP analysis
df = pd.read_csv('../data/processed/features.csv')
features = ["Area", "VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Density", "Region"]
X = df[features]

X_sample = X.sample(n=5000, random_state=42)
X_sample_prep = preprocessor.transform(X_sample)

# Get feature names
cat_names = preprocessor.named_transformers_["cat"].get_feature_names_out()
num_names = preprocessor.transformers_[0][2]
feature_names = list(num_names) + list(cat_names)

## 2. Compute SHAP Values
`TreeExplainer` is an incredibly fast exact algorithm specifically designed for tree-based models like LightGBM.

In [ ]:
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_sample_prep)

## 3. Global Feature Importance
Which features have the largest absolute impact on the model's predictions overall?

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample_prep, feature_names=feature_names, plot_type="bar", max_display=10, show=False)
plt.title("Top Risk Drivers (Mean Absolute SHAP Value)")
plt.tight_layout()
plt.show()

**Insight**: `BonusMalus` is overwhelmingly the most important feature. This aligns perfectly with our EDA findings and confirms that historical driving behavior is the best predictor of future claims.

## 4. Directional Impact (Beeswarm Plot)
How does a high vs low value of a feature impact the prediction?

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample_prep, feature_names=feature_names, max_display=10, show=False)
plt.title("SHAP Beeswarm: Directional Impact on Claim Frequency")
plt.tight_layout()
plt.show()

**Detailed Insights**:
- **BonusMalus**: The red dots (high BonusMalus score / worse driving record) strongly push the prediction to the right (higher claim frequency). Blue dots (good drivers) lower the risk.
- **VehPower**: Higher vehicle power slightly increases risk.
- **DrivAge**: Blue dots (younger ages) tend to push predictions to the right, confirming the young driver risk premium.
- **Area_F**: Living in Area F (urban) increases risk compared to rural areas.

This explainability is highly valuable for underwriters to confidently deploy these ML models into production.